# Ensemble Methods: Bagging, Boosting & Stacking Engineering Lab

Ensemble learning combines diverse base estimators to synthesize a composite hypothesis possessing lower bias, lower variance, or both. This lab implements and analyzes the three major ensemble paradigms: Bootstrap Aggregation (Bagging), Sequential Functional Gradient Descent (Boosting), and Meta-Learning (Stacking).

### Table of Contents
1. [Bagging & Random Forest: Variance Reduction via Tree Decorrelation](#1-bagging--random-forest-variance-reduction-via-tree-decorrelation)
2. [AdaBoost: Adaptive Sample Reweighting](#2-adaboost-adaptive-sample-reweighting)
3. [Modern Gradient Boosted Decision Trees: XGBoost & LightGBM](#3-modern-gradient-boosted-decision-trees-xgboost--lightgbm)
4. [Stacked Generalization (Stacking) with Out-of-Fold Meta-Learners](#4-stacked-generalization-stacking-with-out-of-fold-meta-learners)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

sns.set_theme(style="whitegrid")
print("Loaded Ensemble Methods interactive environment.")

## 1. Bagging & Random Forest: Variance Reduction via Tree Decorrelation
If $B$ independent estimators each have variance $\sigma^2$, their average has variance $\frac{\sigma^2}{B}$. If they have pairwise correlation $\rho$, variance is $\rho \sigma^2 + \frac{1 - \rho}{B} \sigma^2$.
Random Forest reduces $\rho$ by considering only a random subset of $\sqrt{D}$ features at each split.

In [ ]:
# Generate noisy dataset with 20 features
np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=20, n_informative=8, random_state=42)

n_trees_range = [1, 5, 10, 25, 50, 100]
bag_scores = []
rf_scores = []

for n in n_trees_range:
    bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=n, random_state=42).fit(X, y)
    rf = RandomForestClassifier(n_estimators=n, max_features="sqrt", random_state=42).fit(X, y)
    bag_scores.append(accuracy_score(y, bag.predict(X)))
    rf_scores.append(accuracy_score(y, rf.predict(X)))

plt.figure(figsize=(8, 4.5))
plt.plot(n_trees_range, bag_scores, marker="o", label="Standard Bagging (All Features)", color="#d62728")
plt.plot(n_trees_range, rf_scores, marker="s", label="Random Forest (sqrt(D) Subsampling)", color="#1f77b4", lw=2)
plt.title("Ensemble Size vs. Training Accuracy", fontweight="bold")
plt.xlabel("Number of Trees")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

## 2. AdaBoost: Adaptive Sample Reweighting
AdaBoost sequentially boosts weak decision stumps by exponentially scaling sample weights on misclassified observations.

In [ ]:
ada = AdaBoostClassifier(n_estimators=30, random_state=42).fit(X, y)
estimator_errors = ada.estimator_errors_
estimator_weights = ada.estimator_weights_

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(1, len(estimator_errors) + 1), estimator_errors, color="#d95f02", marker="o")
axes[0].set_title("Stump Weighted Error Rate over Iterations", fontweight="bold")
axes[0].set_xlabel("Boosting Stage")
axes[0].set_ylabel("Weighted Error")

axes[1].plot(range(1, len(estimator_weights) + 1), estimator_weights, color="#7570b3", marker="s")
axes[1].set_title("Stump Voting Weight (alpha_m) Decay", fontweight="bold")
axes[1].set_xlabel("Boosting Stage")
axes[1].set_ylabel("Alpha")
plt.tight_layout()
plt.show()

## 3. Modern Gradient Boosted Decision Trees: XGBoost & LightGBM
Gradient Boosting builds trees greedily to minimize arbitrary differentiable loss functions via functional gradient descent.

In [ ]:
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X, y)

# Compute cumulative stage loss
stage_losses = [loss for loss in gb.train_score_]

plt.figure(figsize=(7, 4.5))
plt.plot(stage_losses, color="#2b5c8f", lw=2)
plt.title("Gradient Boosting Negative Binomial Deviance vs. Tree Stages", fontweight="bold")
plt.xlabel("Number of Trees")
plt.ylabel("Loss")
plt.tight_layout()
plt.show()

## 4. Stacked Generalization (Stacking)
Blending diverse base learners (Random Forest, SVM, k-NN) via an out-of-fold Logistic Regression meta-learner.

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

estimators = [
    ("rf", RandomForestClassifier(n_estimators=50, random_state=42)),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
    ("svc", SVC(probability=True, random_state=42))
]

stack = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(), cv=5)
stack.fit(X, y)
print(f"Stacking Ensemble Accuracy on Synthetic Data: {accuracy_score(y, stack.predict(X)):.2%}")